# Variantes de selección del sintético de Generate-then-Refine (GtR)

Prueba si **seleccionar** el sintético de GtR (en vez de usar todo lo aprobado) mejora al baseline. Funciona **igual en tu computador y en Colab**, y **no usa la API de Claude ni gasta créditos**:
todo sale de las cachés de GtR que ya están en el repositorio (`tecnicas_aumento/salidas/en_linea_marcadores_fuentes/generate_then_refine/`).

**Por qué.** GtR 120 aporta un sintético muy desbalanceado (mucho DES, SAL y NEG; poco EMO, AFI y REQUEST) y el de DES, PRG, SAL y NEG empeora a su propia categoría
(ver `evaluacion/diagnostico_baseline/README.md`). Aquí se prueban 10 variantes con el **protocolo B** (C elegido por CV interna K=5, 5 folds congelados), así que las cifras son comparables con los 12 experimentos:

| Variante | Qué hace |
|---|---|
| `ref_sin_aumento`, `ref_gtr_completo` | referencias: baseline y todo lo aprobado de GtR 120 (deben dar ~0.665 y ~0.652 en mBERT; el segundo puede variar ~0.003 entre máquinas) |
| `v1_c40_bal` | 40 sintéticos por clase, balanceado |
| `v2_c40_bal_sim060` | idem con `similitud_labse >= 0.60` |
| `v3_c40_bal_sim060_sinDES` | idem, sin sintético de DES |
| `v4_c20_bal_sim060` | 20 por clase con `similitud_labse >= 0.60` |
| `v5_c20_AFI_PRG_REQUEST_sim060` | solo AFI, PRG y REQUEST |
| `v6_c40_bal_margen` | solo sintéticos más cerca de su clase que de otra por un margen ≥ 0.05 (centroides LaBSE de las reales) |
| `v7_c40_bal_margen_borde` | lo contrario: del lado correcto pero cerca de la frontera (0 < margen < 0.05) |
| `v8_c40_bal_sinDES` | 40 por clase, sin DES y sin filtro de similitud |

**Criterio de éxito:** F1 macro ≥ baseline, DES no cae más de 0.01 y ninguna clase cae más de 0.02.
**Tiempo:** unos 40 a 90 minutos con mBERT (los entrenamientos son CPU; en Colab, activa una GPU para acelerar los embeddings).
**En Colab:** antes haz `git push` del repositorio con las cachés (`salidas/en_linea_marcadores_fuentes/`) y `evaluacion/variantes_gtr.py`.
Si se corta, **vuelve a correr las celdas**: se reanuda sola (y en Colab guarda en tu Drive).

## 1. Preparar el entorno (local o Colab)

In [ ]:
import os, sys, subprocess, time, shutil
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules or os.path.exists("/content/sample_data")
GITHUB_USUARIO = "LhiaHC"   # solo se usa en Colab
USAR_DRIVE = True           # solo en Colab: guarda los resultados en Mi unidad/shiwilu_variantes_gtr (se reanuda si se corta)

if EN_COLAB:
    os.chdir("/content")
    if not Path("/content/shiwilu-tesis").exists():
        subprocess.run(["git", "clone", f"https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git"], check=True)
    os.chdir("/content/shiwilu-tesis")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "scikit-learn", "joblib", "matplotlib", "pandas"], check=True)
else:
    raiz = Path.cwd().resolve()
    while not (raiz / "shiwilu").exists() and raiz != raiz.parent:   # sube hasta la raiz del repositorio
        raiz = raiz.parent
    assert (raiz / "shiwilu").exists(), "Abre el notebook dentro del repositorio shiwilu-tesis."
    os.chdir(raiz)

RAIZ = Path.cwd()
print("Entorno:", "Colab" if EN_COLAB else "local", "| raiz del repositorio:", RAIZ)

SCRIPT = RAIZ / "oe2_aumento_de_datos/evaluacion/variantes_gtr.py"
CACHE = RAIZ / "oe2_aumento_de_datos/tecnicas_aumento/salidas/en_linea_marcadores_fuentes/generate_then_refine"
RESULTADOS = RAIZ / "oe2_aumento_de_datos/evaluacion/resultados/variantes_gtr"
requeridos = [SCRIPT, RAIZ / "corpus/corpus_shiwilu_final.csv", RAIZ / "oe2_aumento_de_datos/particiones/folds_fijos.csv",
              CACHE / "fold0_pool.csv", CACHE / "fold0_pool_l5.csv", CACHE / "fold0_in0.csv", CACHE / "fold4_in4_l5.csv"]
faltan = [str(r.relative_to(RAIZ)) for r in requeridos if not r.exists()]
assert not faltan, "Faltan archivos (haz git push de lo nuevo, o trae las cachés de GtR): " + ", ".join(faltan)
print("Archivos necesarios: OK. Este notebook NO llama a la API.")

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destino = Path("/content/drive/MyDrive/shiwilu_variantes_gtr")
    destino.mkdir(exist_ok=True)
    RESULTADOS.parent.mkdir(parents=True, exist_ok=True)
    if RESULTADOS.is_symlink() or RESULTADOS.exists():
        if RESULTADOS.is_dir() and not RESULTADOS.is_symlink():
            shutil.rmtree(RESULTADOS)
        else:
            RESULTADOS.unlink()
    os.symlink(destino, RESULTADOS)   # cada archivo se escribe directamente en tu Drive
    print("Resultados en Drive:", destino)

try:
    import torch
    print("GPU disponible:", torch.cuda.is_available())
except ImportError:
    pass

## 2. Configuración

- `MODELOS`: por defecto solo mBERT (el del criterio de éxito). Agrega `"labse"` y `"xlmr"` para los tres (tarda bastante más).
- `SEMILLAS`: muestras aleatorias del balanceo (las variantes sin muestreo no cambian). Más semillas = más estable y más lento.
- `EXTRAS`: `True` agrega los umbrales de similitud 0.50, 0.55 y 0.65 y los tamaños 20, 10 y 5 por clase.
- `VARIANTES`: `None` = las 10 principales, o una lista de nombres.

In [ ]:
MODELOS = ["mbert"]      # ["mbert", "labse", "xlmr"] para los tres
SEMILLAS = [0, 1]
EXTRAS = False
VARIANTES = None         # por ejemplo ["ref_sin_aumento", "ref_gtr_completo", "v1_c40_bal"]
N_JOBS = -1              # hilos para evaluar los folds en paralelo
REHACER = False          # True = recalcula aunque ya exista el archivo de una variante

## 3. Correr el experimento (se reanuda si se corta)

In [ ]:
comando = [sys.executable, str(SCRIPT), "--modelos", *MODELOS, "--semillas", *map(str, SEMILLAS), "--n-jobs", str(N_JOBS)]
if VARIANTES: comando += ["--variantes", *VARIANTES]
if EXTRAS: comando += ["--extras"]
if REHACER: comando += ["--rehacer"]
print(" ".join(comando), "\n")

RESULTADOS.mkdir(parents=True, exist_ok=True)
log = RESULTADOS / f"log_{time.strftime('%Y%m%d_%H%M%S')}.txt"
env = {**os.environ, "PYTHONIOENCODING": "utf-8", "TOKENIZERS_PARALLELISM": "false", "MPLBACKEND": "Agg"}
proc = subprocess.Popen(comando, cwd=RAIZ, text=True, encoding="utf-8", errors="replace", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=env)
with open(log, "a", encoding="utf-8") as f:
    for linea in proc.stdout:
        if "Batches:" in linea or "Loading weights" in linea:
            continue
        print(linea, end=""); f.write(linea); f.flush()
proc.wait()
assert proc.returncode == 0, f"El script termino con codigo {proc.returncode}: revisa el log {log}"

## 4. Resultados: F1 macro por variante y criterio de éxito

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

R = pd.read_csv(RESULTADOS / "resumen_variantes.csv")
PC = pd.read_csv(RESULTADOS / "f1_por_clase.csv")
CNT = pd.read_csv(RESULTADOS / "conteos_por_clase.csv")
CLASES = ["AFI", "DES", "EMO", "NEG", "PRG", "REQUEST", "SAL"]

# validacion del montaje: las referencias deben coincidir con los resultados oficiales (mBERT: 0.665 sin aumento y 0.652 con GtR completo)
for m, g in R.groupby("modelo"):
    oficial = {"mbert": (0.665, 0.652), "labse": (0.587, 0.593), "xlmr": (0.570, 0.589)}[m]
    for v, o in zip(["ref_sin_aumento", "ref_gtr_completo"], oficial):
        x = g[g.variante == v]
        if len(x):
            ok = abs(x.f1_macro.iloc[0] - o) < 0.006   # el C de la CV interna es sensible a diferencias numericas minimas entre maquinas
            print(f"[{m}] {v}: {x.f1_macro.iloc[0]:.3f} (oficial {o:.3f}) {'OK' if ok else '*** NO COINCIDE: revisa la configuracion ***'}")

for m, g in R.groupby("modelo"):
    g = g.copy()
    g["dif vs sin aumento"] = [("-" if pd.isna(d) else f"{d:+.3f}{'*' if s else ''}") for d, s in zip(g["dif_vs_sin_aumento"], g["distinguible_de_cero"])]
    g["cumple"] = g["cumple_criterio"].map({True: "SI", False: "no"}).fillna("-")
    print(f"\n=== {m}: F1 macro [* = diferencia con IC95% que no incluye 0] ===")
    display(g.sort_values("f1_macro", ascending=False)[["variante", "f1_macro", "dif vs sin aumento", "dif_DES", "peor_clase", "peor_dif_clase", "cumple"]].round(3).reset_index(drop=True))

In [ ]:
# F1 por variante frente al baseline
for m, g in R.groupby("modelo"):
    g = g.sort_values("f1_macro")
    base = g[g.variante == "ref_sin_aumento"].f1_macro.iloc[0]
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(g) + 1.5))
    colores = ["#2e7d32" if c is True else ("#9e9e9e" if v.startswith("ref_") else "#c62828") for v, c in zip(g.variante, g.cumple_criterio)]
    ax.barh(g.variante, g.f1_macro, color=colores)
    ax.axvline(base, color="k", ls="--", lw=1, label=f"sin aumento = {base:.3f}")
    ax.set_xlim(max(0.5, g.f1_macro.min() - 0.03), g.f1_macro.max() + 0.02); ax.set_xlabel("F1 macro"); ax.set_title(f"{m}: verde = cumple el criterio, rojo = no, gris = referencia")
    ax.legend(); plt.tight_layout(); plt.show()

# cambio del F1 de cada clase frente al baseline (rojo = baja)
for m in R.modelo.unique():
    t = PC[PC.modelo == m].set_index("variante")[CLASES]
    d = t.sub(t.loc["ref_sin_aumento"]).drop(index="ref_sin_aumento")
    print(f"\n{m}: cambio del F1 por clase frente a sin aumento")
    display(d.round(3).style.background_gradient(cmap="RdYlGn", vmin=-0.08, vmax=0.08))

## 5. Cuánto sintético usó cada variante (por clase, en el pool de cada fold)

In [ ]:
display(CNT.round(0))

## 6. Guardar / descargar los resultados

In [ ]:
zip_path = shutil.make_archive(str(RAIZ / "variantes_gtr_resultados"), "zip", RESULTADOS)
print("Archivo .zip:", zip_path)
if EN_COLAB:
    from google.colab import files
    try:
        files.download(zip_path)
    except Exception as e:
        print("La descarga automatica fallo (", e, "). Los resultados tambien estan en tu Drive: Mi unidad/shiwilu_variantes_gtr")
else:
    print("Resultados en:", RESULTADOS)